# Sequence generation

Use DeepD to generate **one** sequence of 100 bp from `Caenorhabditis elegans`,
feeding `ATG` as the prompt (sampling temperature 1.5), then compute the
sequence's GC content.

In [1]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Locate the repository root (contains apiexample/cli.py).
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "apiexample" / "cli.py").is_file():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("cannot locate apiexample/cli.py")
    REPO_ROOT = REPO_ROOT.parent

_notebook_file = "sequence_generation.ipynb"
_here = Path.cwd().resolve() / _notebook_file
if _here.is_file():
    NOTEBOOK_DIR = _here.parent
else:
    NOTEBOOK_DIR = next(REPO_ROOT.rglob(_notebook_file)).resolve().parent

RESULTS_DIR = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

## Configuration


In [2]:
def _read_api_token() -> str:
    token = os.environ.get("INFERENCE_API_TOKEN", "").strip()
    if token:
        return token
    for key_file in (REPO_ROOT / "API-Key.txt", REPO_ROOT / "apiexample" / "API-Key.txt"):
        if not key_file.is_file():
            continue
        for line in key_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#"):
                return line
    raise RuntimeError("set INFERENCE_API_TOKEN or add API-Key.txt at the repo root")

TOKEN = _read_api_token()

SPECIES_NAME = "Caenorhabditis elegans"
PROMPT = "ATG"
N_SEQUENCES = 1
TEMPERATURE = 1.5
TARGET_LEN = 100
# The gateway returns only the generated bases (the prompt is not part of
# the returned text), so request the full target length.
MAX_TOKENS = TARGET_LEN

## Generate


In [3]:
CLI_SCRIPT = REPO_ROOT / "apiexample" / "cli.py"


def _run_cli(cmd: list) -> Path:
    before = {p.name for p in RESULTS_DIR.glob("*.json")}
    env = dict(os.environ, INFERENCE_API_TOKEN=TOKEN)
    proc = subprocess.run(
        [sys.executable, str(CLI_SCRIPT), *cmd],
        capture_output=True,
        text=True,
        timeout=1800,
        env=env,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"cli.py failed:\n{proc.stdout}\n{proc.stderr}")
    m = re.search(r"\[done\] saved:\s*(\S+)", proc.stdout)
    if m:
        return Path(m.group(1))
    newest = [p for p in RESULTS_DIR.glob("*.json") if p.name not in before]
    if newest:
        return max(newest, key=lambda p: p.stat().st_mtime)
    raise FileNotFoundError("cli.py did not produce a result file")


def load_result(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        doc = json.load(f)
    inner = doc.get("result", doc)
    if isinstance(inner, dict) and isinstance(inner.get("result"), dict):
        inner = inner["result"]
    return inner


def run_generate_job(prompt: str) -> str:
    path = _run_cli(
        [
            "--prompt", prompt,
            "--task-type", "generate",
            "--max-tokens", str(MAX_TOKENS),
            "--temperature", str(TEMPERATURE),
            "--top-k", "4",
            "--top-p", "1.0",
            "--species", SPECIES_NAME,
            "--output-dir", str(RESULTS_DIR),
            "--poll-interval", "3",
            "--timeout", "900",
        ]
    )
    return str(load_result(path).get("text", "") or "")


def clean_sequence(text: str) -> str:
    text = str(text or "").upper()
    cleaned = "".join(ch for ch in text if ch in "ACGT")
    return cleaned[:TARGET_LEN]


# Generate and cache a single sequence. A cached row is reused only when it
# was produced with the same prompt and temperature; delete the CSV to force
# a fresh generation.
CACHE_CSV = RESULTS_DIR / "generated_sequences.csv"
data = pd.read_csv(CACHE_CSV) if CACHE_CSV.is_file() else pd.DataFrame()
if not {"tag", "prompt", "temperature", "sequence"}.issubset(data.columns):
    data = pd.DataFrame(columns=["tag", "prompt", "temperature", "sequence"])

tag = "seq_00"
cached = None
rows = data[data["tag"] == tag]
if len(rows) == 1 and str(rows["prompt"].iloc[0]) == PROMPT:
    try:
        if abs(float(rows["temperature"].iloc[0]) - TEMPERATURE) < 1e-9:
            seq = rows["sequence"].iloc[0]
            cached = seq if isinstance(seq, str) else None
    except (TypeError, ValueError):
        cached = None

sequence = cached if cached is not None else clean_sequence(run_generate_job(PROMPT))
data = pd.DataFrame(
    [{"tag": tag, "prompt": PROMPT, "temperature": TEMPERATURE, "sequence": sequence}]
)
data.to_csv(CACHE_CSV, index=False)
print(f"{tag} (temperature={TEMPERATURE}):")
print(sequence)

seq_00 (temperature=1.5):
AGGTTCACCGGATACGCGGTATATGAAGTTCCAGATGGTTGTCAAAATCGACTCAGAAGCCGTAAGCCGAAAAGGCTCTATTCGTTTTTAATATTTACAT

## Compute GC content


In [4]:
data["sequence"] = data["sequence"].str.upper().str[:TARGET_LEN]
data["seq_len"] = data["sequence"].str.len()
data["gc_count"] = data["sequence"].str.count("[GC]")
data["gc_content"] = data["gc_count"] / data["seq_len"]
data[["tag", "sequence", "seq_len", "gc_count", "gc_content"]]


,tag,sequence,seq_len,gc_count,gc_content
0,seq_00,AGGTTCACCGGATACGCGGTATATGAAGTTCCAGATGGTTGTCAAA...,100,41,0.41


## GC content

In [5]:
gc = data["gc_content"].iloc[0]
print(f"GC content: {gc:.2%}")
print(data[["tag", "seq_len", "gc_count", "gc_content"]].to_string(index=False))

GC content: 41.00%
   tag  seq_len  gc_count  gc_content
seq_00      100        41        0.41